# Autodata 3.45 — extractor cloud v4 (RAR fix)

Rezolvă eroarea **Unsupported Method** din p7zip-ul Colab, fără să ceară o parolă. Folosește executabilul oficial **7-Zip 26.05 (7zz)**, verificat SHA256 înainte de utilizare. Nu execută programele extrase.

**Dacă arhiva este deja descărcată complet în acest runtime:** nu rula din nou descărcarea; rulează doar celula de verificare și extragere (a treia celulă cu cod) și apoi celulele de arhivare/încărcare în Drive. Dacă ai pornit o sesiune nouă Colab, folosește **Runtime → Run all**; fișierele din sesiunile Colab închise nu sunt permanente.

Rezultatul va fi în **My Drive/Autodata_3_45_EXTRACTED/** în pachete ZIP de sub 190 MB, plus inventarul complet. Originalul din Drive rămâne intact. Finalizarea este confirmată doar de mesajul **TRANSFER FINALIZAT**.


In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth, os, re, sys, time, json, shutil, hashlib, subprocess, getpass
from pathlib import Path
from google.auth.transport.requests import AuthorizedSession
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

FILE_ID = "1lC8pgAMU4RMkcatQClkzNnABJZpsKxZE"
ROOT = Path("/content/autodata_v2")
ARCHIVE = ROOT / "Autodata__3.45.rar"
EXTRACT = ROOT / "extracted"
PACKS = ROOT / "packages"
OUTPUT_FOLDER_NAME = "Autodata_3_45_EXTRACTED"
CHUNK = 8 * 1024 * 1024
PACK_LIMIT = 180 * 1024 * 1024
ROOT.mkdir(parents=True, exist_ok=True)

credentials, project = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
service = build("drive", "v3", credentials=credentials, cache_discovery=False)
session = AuthorizedSession(credentials)
meta = service.files().get(fileId=FILE_ID, fields="id,name,mimeType,size,md5Checksum,capabilities(canDownload)").execute()
assert meta["name"] == ARCHIVE.name, f"Fișier nepotrivit: {meta['name']}"
assert meta["mimeType"] != "application/vnd.google-apps.folder"
if meta.get("capabilities", {}).get("canDownload") is False:
    raise PermissionError("Contul nu are drept de descărcare.")
EXPECTED = int(meta["size"])
print("Drive API conectat. Arhiva:", meta["name"], f"({EXPECTED/2**30:.2f} GiB)")
print("Spațiu disponibil în Colab:", f"{shutil.disk_usage('/content').free/2**30:.1f} GiB")
if shutil.disk_usage("/content").free + (ARCHIVE.stat().st_size if ARCHIVE.exists() else 0) < EXPECTED + 1024**3:
    raise RuntimeError("Colab nu are spațiu pentru fișierul RAR. Schimbă runtime-ul cu mai mult disk.")

In [ ]:
# V3 — Drive REST API: reluare exactă, răspunsuri HTTP diagnosticate, retry cu backoff.
# Păstrează /content/autodata_v2/Autodata__3.45.rar dacă există o copie parțială.
import requests, random
from google.auth.transport.requests import Request

ENDPOINT = f"https://www.googleapis.com/drive/v3/files/{FILE_ID}"
TRANSIENT_HTTP = {408, 429, 500, 502, 503, 504}
TRANSIENT_403 = {"rateLimitExceeded", "userRateLimitExceeded", "backendError", "sharingRateLimitExceeded"}
MAX_ATTEMPTS = 14

def error_details(response):
    try:
        obj = response.json()
        err = obj.get("error", {})
        reasons = [x.get("reason", "") for x in err.get("errors", [])]
        return reasons[0] if reasons else "unknown", err.get("message", "")[:300]
    except Exception:
        return "unknown", "Răspuns HTTP fără detalii JSON"

def retry_delay(attempt, response=None):
    hint = None if response is None else response.headers.get("Retry-After")
    if hint and hint.isdecimal():
        return min(max(int(hint), 2), 120)
    return min(2 ** min(attempt, 6) + random.random(), 65)

if not ARCHIVE.exists():
    ARCHIVE.touch()
current = ARCHIVE.stat().st_size
if current > EXPECTED:
    raise RuntimeError("Copia parțială este mai mare decât sursa. Oprește și verifică fișierul.")
print(f"RELUARE de la {current}/{EXPECTED} octeți ({current/EXPECTED:.1%}).")
print("Răspunsurile HTTP temporare vor fi reîncercate fără pierderea progresului.")
with ARCHIVE.open("ab") as output:
    while current < EXPECTED:
        end = min(current + CHUNK, EXPECTED) - 1
        completed = False
        last_error = ""
        for attempt in range(MAX_ATTEMPTS):
            wait = None
            try:
                with session.get(ENDPOINT, params={"alt": "media"},
                                 headers={"Range": f"bytes={current}-{end}"},
                                 stream=True, timeout=(30, 180)) as response:
                    code = response.status_code
                    if code == 206:
                        header = response.headers.get("Content-Range", "")
                        expected_prefix = f"bytes {current}-{end}/{EXPECTED}"
                        if header.strip() != expected_prefix:
                            raise RuntimeError(
                                f"Content-Range nu corespunde: {header!r}; așteptat {expected_prefix!r}")
                        data = b"".join(response.iter_content(chunk_size=1024 * 1024))
                        if len(data) != end - current + 1:
                            raise IOError(f"Răspuns incomplet: {len(data)} octeți")
                        output.write(data)
                        output.flush()
                        current += len(data)
                        completed = True
                    else:
                        reason, message = error_details(response)
                        last_error = f"HTTP {code}, reason={reason}, message={message}"
                        print(f"Răspuns Google Drive: {last_error}")
                        if code == 403:
                            if reason in {"downloadQuotaExceeded", "dailyLimitExceeded", "storageQuotaExceeded"}:
                                raise PermissionError(
                                    "Google Drive indică o limită de cotă: "+last_error+
                                    f". Copia parțială ({current}/{EXPECTED} octeți) rămâne în /content; "
                                    "poate fi reluată după resetarea cotei, dacă runtime-ul Colab mai există.")
                            if reason not in TRANSIENT_403:
                                raise PermissionError(
                                    "Google Drive a refuzat cererea; nu este o eroare de rețea temporară. "+last_error)
                        elif code == 401:
                            credentials.refresh(Request())
                        elif code == 200:
                            raise RuntimeError(
                                "Drive a ignorat antetul Range (HTTP 200). "
                                "Nu pot anexa în siguranță întregul fișier la copia parțială. "+last_error)
                        elif code not in TRANSIENT_HTTP:
                            raise RuntimeError("Eroare permanentă la Drive API: "+last_error)
                        wait = retry_delay(attempt, response)
            except PermissionError:
                raise
            except (requests.RequestException, IOError) as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                print(f"Întrerupere rețea, încercarea {attempt+1}/{MAX_ATTEMPTS}: {last_error[:140]}")
                wait = retry_delay(attempt)
            if completed:
                break
            if wait is not None and attempt + 1 < MAX_ATTEMPTS:
                print(f"Reîncerc de la octetul {current} peste {wait:.0f} secunde...")
                time.sleep(wait)
        if not completed:
            raise RuntimeError(
                f"Nu s-a putut descărca segmentul {current}-{end} după {MAX_ATTEMPTS} încercări. "
                f"Ultima eroare: {last_error}. "
                f"Copia parțială este păstrată ({current/EXPECTED:.1%}).")
        if current % (128*2**20) < CHUNK or current == EXPECTED:
            print(f"Descărcare: {current/EXPECTED:.1%}")
assert ARCHIVE.stat().st_size == EXPECTED, "Descărcarea nu este completă."
md5 = hashlib.md5()
sha = hashlib.sha256()
with ARCHIVE.open("rb") as inp:
    for block in iter(lambda: inp.read(16 * 1024 * 1024), b""):
        md5.update(block)
        sha.update(block)
if meta.get("md5Checksum") and meta["md5Checksum"].lower() != md5.hexdigest().lower():
    raise RuntimeError("MD5 diferit de valoarea din Google Drive. Copia este invalidă.")
print("DOWNLOAD_VERIFIED=PASS SHA256:", sha.hexdigest())

In [ ]:
# V4 — remediere "Unsupported Method" la RAR: 7-Zip oficial 26.05 (7zz).
# NU re-descărcăm Autodata__3.45.rar: se folosește ARCHIVE deja existent în /content.
import hashlib, tarfile, io, platform, requests, re, shutil, subprocess, os
from pathlib import Path, PurePosixPath

if not ARCHIVE.exists() or ARCHIVE.stat().st_size != EXPECTED:
    raise RuntimeError(
        "Arhiva nu există complet în runtime-ul curent. Execută celula de descărcare de mai sus "
        "înaintea extragerii. Nu șterge fișierul parțial, descărcarea poate continua.")
print("ARHIVA LOCALĂ GĂSITĂ:", ARCHIVE, ARCHIVE.stat().st_size, "bytes")
print("Verific 7-Zip 26.05 cu decodor RAR modern (nu p7zip din apt).")

architecture = platform.machine().lower()
if architecture not in ("x86_64", "amd64"):
    raise RuntimeError("Acest notebook necesită Colab Linux x86-64; arhitectură actuală: "+architecture)

BIN = ROOT / "7zz"
BUNDLE = ROOT / "7z2605-linux-x64.tar.xz"
OFFICIAL_URLS = [
    "https://github.com/ip7z/7zip/releases/download/26.05/7z2605-linux-x64.tar.xz",
    "https://www.7-zip.org/a/7z2605-linux-x64.tar.xz"
]
OFFICIAL_SHA256 = "22cf82fcc6eb7474f1d5bb4f02c0f8ebd573a718089d6833dc30f8350a3b232e"

if not BIN.is_file():
    if not BUNDLE.exists() or hashlib.sha256(BUNDLE.read_bytes()).hexdigest() != OFFICIAL_SHA256:
        last_error=None
        for url in OFFICIAL_URLS:
            try:
                print("Preiau 7-Zip oficial:",url)
                resp=requests.get(url,timeout=(30,120))
                resp.raise_for_status()
                content=resp.content
                received_sha=hashlib.sha256(content).hexdigest()
                if received_sha != OFFICIAL_SHA256:
                    raise ValueError("SHA256 diferit pentru descărcarea 7-Zip.")
                BUNDLE.write_bytes(content)
                last_error=None
                break
            except Exception as e:
                last_error=str(e)
                print("Nu s-a putut prelua de la această adresă:", last_error[:200])
        if last_error:
            raise RuntimeError("Nu am putut descărca utilitarul oficial: "+last_error)

    with tarfile.open(BUNDLE, "r:xz") as tar:
        candidates=[member for member in tar.getmembers()
                    if Path(member.name).name=="7zz" and member.isfile()]
        if len(candidates)!=1:
            raise RuntimeError("Arhiva 7-Zip nu conține exact un executabil 7zz.")
        obj=tar.extractfile(candidates[0])
        if obj is None:
            raise RuntimeError("Nu se poate citi 7zz.")
        BIN.write_bytes(obj.read())
    BIN.chmod(0o755)

ver=subprocess.run([str(BIN)],capture_output=True,text=True,errors="replace",timeout=20)
print("Utilitar:",(ver.stdout+ver.stderr).splitlines()[0:3])
if "26.05" not in (ver.stdout+ver.stderr):
    raise RuntimeError("Versiunea 7zz nu este cea așteptată (26.05).")

def sevenzip(action):
    args=[str(BIN),action,"-y",str(ARCHIVE)]
    if action=="x": args.insert(3,"-o"+str(EXTRACT))
    log=ROOT/("sevenzip_"+action+".log")
    with log.open("w") as output:
        code=subprocess.run(args,stdin=subprocess.DEVNULL,stdout=output,
                            stderr=subprocess.STDOUT,timeout=7200).returncode
    return code,log

print("Testez integral arhiva cu 7zz 26.05...")
rc,log=sevenzip("t")
if rc!=0:
    tail=log.read_text(errors="replace")[-5000:]
    print("ERORI ÎN ARHIVĂ:",tail)
    if "Unsupported Method" in tail:
        raise RuntimeError("Metoda RAR rămâne incompatibilă chiar cu 7zz modern. Salvează jurnalul pentru analiză.")
    if "Wrong password" in tail or "password" in tail.lower():
        raise RuntimeError("Arhiva raportează criptare, deși nu se așteaptă parolă; verifică jurnalul.")
    raise RuntimeError("Testul de integritate a eșuat. Raport: "+str(log))

print("RAR_TEST=PASS")
listing=subprocess.run([str(BIN),"l","-slt",str(ARCHIVE)],capture_output=True,
                       text=True,errors="replace",timeout=1800,check=True).stdout
(ROOT/"listare_7z.txt").write_text(listing,encoding="utf-8")
body=listing.split("----------",1)[-1]
estimated=sum(int(n) for n in re.findall(r"(?m)^Size = (\d+)$",body))
if estimated>22*1024**3:
    raise RuntimeError(f"Arhiva s-ar extinde la {estimated/2**30:.2f} GiB; peste limita de 22 GiB.")
for filename in re.findall(r"(?m)^Path = (.+)$",body):
    name=filename.replace("\\","/")
    if name.startswith("/") or re.match(r"^[A-Za-z]:",name) or ".." in PurePosixPath(name).parts:
        raise RuntimeError("Cale nesigură în arhivă: "+name[:200])
if shutil.disk_usage("/content").free < estimated + 500*1024**2:
    raise RuntimeError(f"Nu există destul spațiu pentru extragere ({estimated/2**30:.2f} GiB).")
EXTRACT.mkdir(parents=True,exist_ok=True)
rc,log=sevenzip("x")
if rc:
    print(log.read_text(errors="replace")[-5000:])
    raise RuntimeError("7zz nu a putut extrage arhiva. Vezi jurnalul "+str(log))
print("EXTRACT=PASS — verifică apoi celulele pentru inventar și upload în Drive.")

In [ ]:
# Pachetizare în ZIP-uri <190 MB și manifest: conectorul Drive poate citi fiecare pachet.
# Folosim ZIP_STORED pentru mărimi previzibile și pentru a evita recompresia inutilă.
import zipfile
PACKS.mkdir(exist_ok=True)
for old in PACKS.glob("autodata_part_*.zip"): old.unlink()
entries=[]
for p in sorted(EXTRACT.rglob("*")):
    if not p.is_file() or p.is_symlink():
        continue
    relative=p.relative_to(EXTRACT).as_posix()
    entries.append((p, relative, p.stat().st_size))
if not entries: raise RuntimeError("Nu s-a extras niciun fișier.")
if len(entries)>150000: raise RuntimeError("Număr prea mare de fișiere; e necesară altă strategie.")
packing=[]
i=0; used=0; z=None; packname=None
def next_pack():
    global i,used,z,packname
    if z: z.close()
    i+=1;used=0
    packname=f"autodata_part_{i:04d}.zip"
    z=zipfile.ZipFile(PACKS/packname,"w",compression=zipfile.ZIP_STORED,allowZip64=True)
    print("Construiesc:",packname)
try:
    next_pack()
    for source, relative, size in entries:
        # Fișierele mari sunt împărțite în segmente care încap într-un ZIP.
        if size == 0:
            if used + 4096 > PACK_LIMIT: next_pack()
            z.writestr(relative,b"");used+=4096
            packing.append({"path":relative,"size_bytes":0,"part":packname})
            continue
        chunks=[]
        with source.open("rb") as f:
            index=0
            while True:
                buf=f.read(min(64*1024*1024, size - index*64*1024*1024))
                if not buf: break
                name=relative if size<=64*1024*1024 else f"{relative}.chunk{index:04d}"
                if used + len(buf) + 4096 > PACK_LIMIT: next_pack()
                z.writestr(name,buf)
                used+=len(buf)+4096
                chunks.append({"package":packname,"entry":name,"bytes":len(buf)})
                index+=1
        packing.append({"path":relative,"size_bytes":size,"chunks":chunks})
finally:
    if z: z.close()
packages=sorted(PACKS.glob("autodata_part_*.zip"))
for package in packages:
    assert package.stat().st_size < 190*1024*1024, "Un pachet a depășit plafonul."
manifest={"archive":ARCHIVE.name,"original_bytes":EXPECTED,"sha256":sha.hexdigest(),"file_count":len(packing),"package_count":len(packages),"files":packing}
(ROOT/"INVENTAR.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
(ROOT/"INVENTAR.txt").write_text("\n".join(f"{x['size_bytes']:>14} {x['path']}" for x in packing))
print("FILES=",len(entries),"PACKAGES=",len(packages))

In [ ]:
# Upload direct prin Drive API, fără drive.mount(), cu retry și posibilitate de reluare la rerulare.
from googleapiclient.errors import HttpError
def api_call(request):
    for attempt in range(8):
        try: return request.execute(num_retries=4)
        except HttpError as e:
            status=e.resp.status
            if status not in (429,500,502,503,504) and not (status==403 and b"rateLimitExceeded" in e.content):
                raise
            time.sleep(min(3*2**attempt,60))
    raise RuntimeError("Google Drive API: prea multe erori la operație.")
def drive_query(q):
    return api_call(service.files().list(q=q,fields="nextPageToken,files(id,name,size,mimeType)",pageSize=100,spaces="drive")).get("files",[])
matches=drive_query("name = '"+OUTPUT_FOLDER_NAME+"' and mimeType = 'application/vnd.google-apps.folder' and trashed = false")
folder_id = matches[0]["id"] if matches else api_call(service.files().create(
    body={"name":OUTPUT_FOLDER_NAME,"mimeType":"application/vnd.google-apps.folder"},
    fields="id")).get("id")
print("Folder destinație:",f"https://drive.google.com/drive/folders/{folder_id}")
def upload_one(localfile):
    # Evită duplicatele și reîncărcarea fișierelor deja finalizate în Drive.
    fname=localfile.name.replace("'","\\'")
    existing=drive_query(f"name = '{fname}' and '{folder_id}' in parents and trashed = false")
    for item in existing:
        if int(item.get("size","-1"))==localfile.stat().st_size:
            print("Există deja:",localfile.name)
            return item["id"]
    media=MediaFileUpload(str(localfile),mimetype="application/zip" if localfile.suffix==".zip" else "application/octet-stream",
                          chunksize=8*1024*1024,resumable=True)
    task=service.files().create(body={"name":localfile.name,"parents":[folder_id]},media_body=media,fields="id,name,size")
    result=None
    while result is None:
        progress,result=task.next_chunk(num_retries=7)
    print("UPLOAD OK:",localfile.name)
    return result["id"]
for source in [ROOT/"INVENTAR.json", ROOT/"INVENTAR.txt",ROOT/"listare_7z.txt"]+packages:
    upload_one(source)
# Marcare succes după ce toate pachetele au fost încărcate.
done=ROOT/"TRANSFER_FINALIZAT.txt"
done.write_text(f"PASS\nArchive SHA256: {sha.hexdigest()}\nPachete: {len(packages)}\nFișiere: {len(entries)}\n")
upload_one(done)
print("\nTRANSFER FINALIZAT. Verifică în Google Drive folderul Autodata_3_45_EXTRACTED")
print(f"https://drive.google.com/drive/folders/{folder_id}")